# Task 3 — Gender G-N64 width screen

**Run All trains only folds 0 and 4**, from scratch. This screen tests whether a narrower final layer can reduce the clean training–validation gap while keeping validation performance within the agreed limits. It is a capacity trade-off test, not a claim that accuracy will improve.

Use a **fresh Colab L4 runtime** matching 04w. Publish this notebook and its source changes before running. The completed 04w diagnostic and all saved G2/E6 source runs must be on Drive.


## 1. Colab GPU and repository


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


## 2. Teacher data and canonical split


In [ ]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


## 3. Frozen change and evidence checks

Change the final convolution from **256 to 64 channels**; the following BatchNorm and classifier shrink to match. Widths become `[32, 64, 128, 64]` and the parameter count falls from **390,181 to 167,653**.

Keep G2's full images, fold-training RGB statistics, ±2-pixel translation, fixed GeM p=3, plain cross-entropy, AdamW weight decay 0.0001, learning rate and schedule, batch 128, 30 epochs, seed 2753 and final-epoch checkpoint. Parent checkpoints are comparison evidence; the child starts with new random weights. No pretrained weights, new split, extra augmentation, stronger decay or early stopping.

Keep the runtime-default training precision reproduced in 04w. For the decision, reevaluate both saved G2/E6 and the completed child checkpoints in **full IEEE FP32**, using batch 128 and the same images. These separate snapshots include clean training, validation and all standard corruption predictions. Historical registered scores remain separate. This common evaluation setting is not a second training-factor change.

The source check below does not train. It verifies the completed precision evidence and all ten G2/E6 bundles before the first fit. Reusable IEEE snapshots are bound to source, data, code and runtime hashes.


In [ ]:
from fashion.train.task3_gender_narrow import (
    check_gender_narrow_sources, run_gender_narrow_screen,
)

G2_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_v2_g2_translation/gender"
E6_DIR = DRIVE_TASK_DIR / "experiments/t3_gender_e6_gem_p3/gender"
PRECISION_DIR = DRIVE_TASK_DIR / "diagnostics/gender_precision/20260905T085822668071Z"
sources, classes, spec, evidence = check_gender_narrow_sources(
    g2_directory=G2_DIR, e6_directory=E6_DIR, source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR, root=REPO_DIR,
)
print("Source runs:", {name: len(runs) for name, runs in sources.items()})
print("Frozen recipe:", spec.to_dict())
print("Output:", DRIVE_TASK_DIR / spec.artifact_dir / "gender")


## 4. Agreed screen rules

All rules must pass, using the same full-FP32 evaluation for each model:

- Pooled validation macro-F1 falls by at most **0.030 versus matched G2**. The paired whole-family bootstrap 95% lower bound for the difference must be **at least −0.030**. Neither fold may lose more than 0.030. Use 10,000 draws within folds, seed 2753.
- The mean clean training–validation F1 gap falls by at least **0.050**. **Both folds' gaps must shrink.** Use clean evaluation-mode training scores from each finished checkpoint, not online augmented training scores.
- Preserve the existing stricter class guard: no pooled class loses more than **0.020 F1**. Thus the overall 0.030 allowance does not override a class failure. NLL may rise by at most 0.020 and ECE by at most 0.010 versus G2; these measure the quality of model confidence.
- Preserve corruption guards versus matched E6: the translation-induced F1 change improves by at least 0.030; every other standard corruption, including darkening, worsens by at most 0.020. Each corrupted score is measured relative to that model's clean score.
- Exactly **167,653 parameters** and peak allocated GPU memory **strictly below 3,000,000,000 bytes**. Training time and latency are reported without speed caps. A memory failure stops before another fold begins.

The thresholds are derived from the new matched IEEE reference scores; do not substitute the rounded historical values. A pass means this screen met the chosen trade-off, not that the model is accepted or more accurate.


In [ ]:
result = run_gender_narrow_screen(
    g2_directory=G2_DIR, e6_directory=E6_DIR, source_registry_path=DRIVE_REGISTRY,
    precision_directory=PRECISION_DIR, output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY, registry_mirrors=(LOCAL_REGISTRY,), root=REPO_DIR,
)
print("Screen:", result["status"])
for row in result.get("folds", []):
    print("Fold", row["fold"], "train F1:", row["candidate_train_f1"],
          "validation F1:", row["candidate_validation_f1"], "gap:", row["candidate_gap"])
for gate in result.get("checks", []):
    if gate["status"] != "pass":
        print(gate)
if "reason" in result:
    print(result["reason"])


## 5. Stop and review

Do not auto-run folds 1–3, refit or open the held-out test. Tell me when this finishes so the saved result can be checked. Each training run appends a registry row. A completed run is reused only after registry, configuration, lineage and artifact checks.

Read `screen_decision.json`, `clean_gap_comparison.csv`, `ieee_oof_predictions.csv`, `source_audit.json` and `comparison_ieee/` in `experiments/t3_gender_narrow64/gender`. Each IEEE snapshot includes an evaluation manifest with hashes, full probabilities, per-class metrics and corruption scores. The original fold folders and registry keep the native training-run evidence and historical precision scores. Report both the training fit and validation result; a smaller gap alone is not proof of better generalization.
